# 04 — Classical baseline (Stage C, Arm 1)

Logistic regression and gradient boosting on the ten hand-crafted quality features, one model per
output (BLR BRT DRK FRM OBS ROT at >=2 votes, plus UNREC = unrecognizable >=2), at two feature
resolutions (long edge 224 and 512). All modelling logic lives in `src/models/`; this notebook only
calls it and displays results. It regenerates `reports/results_classical.md` and
`reports/results_classical.csv`.

**Partitions:** models are fitted on train and scored on val. The test partition is never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`,
`classical_features_{224,512}.csv` (from `python scripts/extract_classical_features.py DATA_DIR`)
and `image_properties.csv` (from `scripts/extract_image_properties.py`). No GPU needed; the whole
notebook runs in a few minutes on a CPU.

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_classical.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_classical.csv"
PREDICTIONS_DIR = DATA_DIR / "predictions"

from src.data.splits import MODELLED_FLAWS, SEED, get_splits
from src.eda import image_props as ip
from src.eda import labels as lb
from src.eda import report as rp
from src.features.classical import FEATURE_NAMES
from src.models import classical as cm
from src.models import metrics as mt

random.seed(SEED)
np.random.seed(SEED)
RESOLUTIONS = (224, 512)
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed: scikit-learn models on precomputed features",
    seed_note="gradient boosting random_state; logistic regression is deterministic")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

## 1. Data
Targets come from the annotation records via `src/data/splits.py`; features are joined by image
name. Every train and val image must have a feature row at both resolutions.

In [ ]:
splits = get_splits(DATA_DIR)
targets = {p: cm.build_targets(splits[p]) for p in ("train", "val")}
features = {res: cm.load_features(DATA_DIR, res) for res in RESOLUTIONS}
for res, df in features.items():
    for p, t in targets.items():
        missing = t.index.difference(df.index)
        assert missing.empty, f"{len(missing)} {p} images missing from the {res}px features"

X = {(res, p): features[res].loc[targets[p].index] for res in RESOLUTIONS for p in targets}
sections = []  # (heading, markdown) pairs collected for the report

positives = pd.DataFrame({p: t.sum() for p, t in targets.items()})
positives["val %"] = (positives["val"] / len(targets["val"]) * 100).round(1)
print({p: len(t) for p, t in targets.items()})
display(positives)
sections.append(("Data", (
    f"Train {len(targets['train']):,} images (fitting), val {len(targets['val']):,} images (scoring). "
    "Positives per output at >=2 votes:\n\n" + rp.df_to_markdown(positives))))

## 2. Feature extraction summary
Counts are recomputed from the feature CSVs. The run details (runtime, verification) are recorded
from the extraction run itself.

In [ ]:
uniform = features[224]["lum_std"] == 0
extraction = pd.DataFrame({
    f"{res}px": {"rows": len(df), "upsampled (native long edge < target)": int(df["upsampled"].sum()),
                 "upsampled %": round(df["upsampled"].mean() * 100, 1)}
    for res, df in features.items()})
display(extraction)
print("fully uniform images (lum_std = 0):", int(uniform.sum()))
sections.append(("Feature extraction", (
    "Features from `scripts/extract_classical_features.py` on the original images (PIL, no EXIF "
    "rotation), resized to a fixed long edge with aspect preserved and no padding. One code path, "
    "both resolutions.\n\n" + rp.df_to_markdown(extraction) + "\n\n"
    f"- {int(uniform.sum())} images are fully uniform (luminance std 0).\n"
    "- Run: 2026-10-10 on a Colab CPU runtime, single process, about 50 minutes for 27,306 images at "
    "both resolutions (~560 images/min).\n"
    "- Verification (runner check, 26/26 passed): exact train+val image set with no duplicates and "
    "no test images; native sizes equal `image_properties.csv` (no EXIF rotation); upsampled flag "
    "consistent; all values finite and in range; five random images recomputed from the original "
    "zips match their rows exactly.")))

## 3. Fully uniform images
Images with zero luminance variation are blank frames (black, white or flat). They should be
labelled mostly DRK, BRT or UNREC; if they are not, the labels and features disagree.

In [ ]:
labelled = pd.concat([targets["train"], targets["val"]])
u = features[224].loc[uniform, ["lum_mean"]].join(labelled)
display(u.sort_values("lum_mean"))
uniform_rates = (u[cm.OUTPUTS].mean() * 100).round(0).to_frame("% of uniform images positive").T
display(uniform_rates)
sections.append(("Fully uniform images", (
    f"{len(u)} images with luminance std 0. Share positive per output:\n\n"
    + rp.df_to_markdown(uniform_rates, index=False))))

## 4. Train
Four configurations (2 resolutions x 2 model kinds), seven models each. Every fitted component,
scaler included, sees train only; val is only scored. Val scores are saved to
`DATA_DIR/predictions/` (data, not committed) for the Stage F comparison.

In [ ]:
models, scores = {}, {}
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
for res in RESOLUTIONS:
    for kind in cm.KINDS:
        print(f"{kind} @ {res}px")
        models[res, kind] = cm.train_models(X[res, "train"], targets["train"], kind)
        scores[res, kind] = cm.predict_scores(models[res, kind], X[res, "val"])
        scores[res, kind].to_csv(PREDICTIONS_DIR / f"classical_{kind}_{res}_val.csv")
print("val scores saved to", PREDICTIONS_DIR)

## 5. Results on val
AP is the primary metric and is threshold-free; **read it against the base rate** (the AP of a
random scorer). Precision, recall and F1 are at a fixed 0.5 threshold. The macro row averages the
six flaws; UNREC is the separate recognisability head.

In [ ]:
results = {cfg: mt.evaluate(targets["val"], s) for cfg, s in scores.items()}
for (res, kind), table in results.items():
    print(f"\n{kind} @ {res}px")
    display(table)

def config_name(cfg):
    res, kind = cfg
    return f"{kind.upper()} @ {res}"

base = results[224, "lr"]["base rate"]
ap = pd.DataFrame({config_name(c): t["AP"] for c, t in results.items()})
ap.insert(0, "base rate", base)
f1 = pd.DataFrame({config_name(c): t["F1"] for c, t in results.items()})
print("\nAP by configuration")
display(ap)
print("F1 @ 0.5 by configuration")
display(f1)

long = pd.concat({config_name(c): t for c, t in results.items()}, names=["config", "output"])
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
long.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")

sections.append(("Results on val: AP", (
    "Average precision per output. The base rate is the AP of a random scorer.\n\n"
    + rp.df_to_markdown(ap.round(3)))))
sections.append(("Results on val: F1 at threshold 0.5", rp.df_to_markdown(f1.round(3))))
for cfg, table in results.items():
    sections.append((f"Full table: {config_name(cfg)}", rp.df_to_markdown(table.round(3))))

## 6. ROT against the shape-only reference
Landscape-shaped raw pixels predict ROT without any image content (Stage B: AP 0.258 on train).
The same reference is computed here on val, and ROT AP is also split by EXIF orientation:
within orientation 1 the shape cue carries no signal, so that row isolates content.

In [ ]:
props = ip.load_properties(DATA_DIR / "image_properties.csv")
val_props = props[props["image"].isin(targets["val"].index)]
shape_ref = ip.rot_shortcut_scores(val_props, lb.vote_matrix(splits["val"]))
display(shape_ref)

orientation = props.set_index("image").loc[targets["val"].index, "orientation"]
rot_rows = {}
for cfg, s in scores.items():
    row = {}
    for tag in (1, 6):
        idx = orientation.index[orientation == tag]
        y = targets["val"].loc[idx, "ROT"]
        row[f"AP, orientation {tag} (n={len(idx)}, base {y.mean():.3f})"] = mt.average_precision(y, s.loc[idx, "ROT"])
    rot_rows[config_name(cfg)] = row
rot_by_orientation = pd.DataFrame(rot_rows).T
display(rot_by_orientation)
sections.append(("ROT against the shape-only reference (val)", (
    "Geometry-only cues scored as ROT predictors on val:\n\n" + rp.df_to_markdown(shape_ref)
    + "\n\nClassical ROT AP by EXIF orientation:\n\n" + rp.df_to_markdown(rot_by_orientation.round(3)))))

## 7. BLR on natively small images
At 512px, images whose original long edge is below 512 were upsampled, so they carry no extra
high-frequency detail. BLR AP is split by native size to check the 512 configuration on them.

In [ ]:
native_long = features[224].loc[targets["val"].index, ["native_width", "native_height"]].max(axis=1)
blr_rows = {}
for cfg, s in scores.items():
    row = {}
    for label, mask in (("native < 512", native_long < 512), ("native >= 512", native_long >= 512)):
        y = targets["val"].loc[mask, "BLR"]
        row[f"{label} (n={int(mask.sum())}, base {y.mean():.3f})"] = mt.average_precision(y, s.loc[y.index, "BLR"])
    blr_rows[config_name(cfg)] = row
blr_by_size = pd.DataFrame(blr_rows).T
display(blr_by_size)
sections.append(("BLR by native resolution (val)", rp.df_to_markdown(blr_by_size.round(3))))

## 8. Single-flaw check
AP for each flaw on images carrying no other modelled flaw (positives: that flaw alone; negatives:
no flaw). If DRK or BRT keep their AP here, the classical score measures exposure rather than
co-occurring blur.

In [ ]:
single = {config_name(c): mt.single_flaw_ap(targets["val"], s) for c, s in scores.items()}
first = next(iter(single.values()))
single_table = first[["n", "positives (flaw alone)", "base rate"]].copy()
for name, t in single.items():
    single_table[f"AP {name}"] = t["AP"]
display(single_table)
sections.append(("Single-flaw check (val)", rp.df_to_markdown(single_table.round(3))))

## 9. What logistic regression relies on
Weights on standardised features (log1p for the heavy-tailed ones), 224px. Sign is the direction,
magnitude the strength. Sanity check: DRK should lean on low `lum_mean` / high `clip_dark_frac`,
BRT the opposite, BLR on low sharpness.

In [ ]:
coefs = cm.lr_coefficients(models[224, "lr"])
display(coefs.style.background_gradient(cmap="RdBu_r", axis=None, vmin=-abs(coefs).max().max(),
                                        vmax=abs(coefs).max().max()).format("{:.2f}"))
sections.append(("Logistic regression weights (224px, standardised features)",
                 rp.df_to_markdown(coefs.round(2))))

## 10. Write the report
The findings block in `reports/results_classical.md` is hand-written after reading the results and
is preserved on every rerun.

In [ ]:
out = rp.write_report(REPORT_PATH, sections, title="Stage C: Classical baseline (Arm 1)",
                      environment=ENVIRONMENT, findings_heading="Findings (hypothesis check)")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)